# FIBO 2 Turbo — self-hosted example

This notebook runs Bria's **FIBO 2 Turbo** image editing on your own GPU: it asks the Bria Engine for a short-lived package credential, installs **`fibo-2-turbo`** from Bria's **`bria-fibo-2-turbo`** repository, and edits an image. The model weights are downloaded from Bria on the first `setup()`.

FIBO 2 Turbo edits images only. To generate an image from a prompt, use FIBO 2 (the `fibo-2` package).

**Prerequisites**

- A Bria API key in `BRIA_API_TOKEN` — any key of an account that holds a self-hosting license for FIBO 2 Turbo (https://platform.bria.ai/outpost/licenses?model=fibo-2-turbo).
- Python 3.11 or 3.12, and an NVIDIA GPU with at least 32 GB of memory.
- No LLM: the edit in section 4 sends a ready VGL that carries its instruction. A free-form `prompt` needs one.


## 1. Package credential

Request a short-lived credential for the **`bria-fibo-2-turbo`** PyPI repository. The Engine issues it only for a model your account holds a license for; a `403` with `not_entitled` means the account has no license for FIBO 2 Turbo.


In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.request import Request, urlopen

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN", "").strip()  # a key from a secret file ends in a newline
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

ENGINE = os.environ.get("BRIA_CLIENT_BASE_URL", "https://engine.prod.bria-api.com")
url = f"{ENGINE}/v2/outpost/access/code_artifact?repository=bria-fibo-2-turbo"
request = Request(url, headers={"api_token": BRIA_API_TOKEN})

try:
    with urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    body = exc.read().decode("utf-8", errors="replace")
    try:
        code = json.loads(body).get("error", {}).get("message")
    except json.JSONDecodeError:
        code = None
    if code in ("not_entitled", "entitlement_revoked"):
        raise RuntimeError(
            "This account has no active license for FIBO 2 Turbo. Get one at https://platform.bria.ai/outpost/licenses?model=fibo-2-turbo"
        ) from exc
    raise RuntimeError(f"Package credential request failed with HTTP {exc.code}: {body or exc.reason}") from exc

result = payload.get("result") or {}
CODE_ARTIFACT_PASSWORD = (result.get("authorization_token") or "").strip()
if not CODE_ARTIFACT_PASSWORD:
    raise RuntimeError(f"The response did not include an authorization token: {payload}")
print("Package credential acquired. Expires:", result.get("expiration"))

## 2. Install dependencies

Install **`fibo-2-turbo`** from the **`bria-fibo-2-turbo`** repository, with the fibo-2 packages it runs on. Everything else comes from PyPI.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# pip calls os.getcwd(): if Jupyter's working folder was deleted, move to the home folder first.
try:
    Path.cwd()
except FileNotFoundError:
    os.chdir(Path.home())

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
fibo_2_turbo_index = (
    "https://aws:"
    + encoded_password
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-fibo-2-turbo/simple/"
)


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(install_cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install(
    [sys.executable, "-m", "pip", "install", "--upgrade", "fibo-2-turbo>=0.1.0", "--extra-index-url", fibo_2_turbo_index]
)

## 3. Check the license and load the model

`setup()` first checks your license with the Bria Engine. If the license is missing or has ended, `setup()` raises `LicenseError` with a link to your licenses page and loads nothing. Then it checks the LLM setting: if `FIBO_2_TURBO_VGL_LLM` is set and the credentials of its provider are missing, `setup()` raises `ValueError` before it downloads the FIBO 2 Turbo weights. With no LLM at all, it logs a warning that free-form prompts are off, and the edit in section 4 still works. Then it downloads the FIBO 2 Turbo weights from Bria into `~/.bria/models/fibo-2-turbo/<version>/` (set `BRIA_HOME` to move them) and verifies them; later runs reuse them. Last, it loads the model onto the GPU and runs one small warm-up.

If the install cell changed core runtime packages, restart the kernel, run the cell of section 1 again (it sets `BRIA_API_TOKEN`), then run this cell.


In [ ]:
import logging
from pathlib import Path

from common.outpost import LicenseError

from fibo_2_turbo import Fibo2Turbo, Fibo2TurboConfig, Fibo2TurboInput

logging.basicConfig(level=logging.INFO)

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

pipeline = Fibo2Turbo(config=Fibo2TurboConfig(api_token=BRIA_API_TOKEN))
try:
    pipeline.setup()
except LicenseError as error:
    raise RuntimeError(f"{error} ({error.code})") from None
print("FIBO 2 Turbo pipeline setup complete.")


## 4. Edit an image

`execute()` takes one `Fibo2TurboInput`: 1 to 4 PIL images and the instruction of the edit. It returns the edited image, and the VGL that the model read as `result.vgl`. The instruction below is a ready VGL that carries an `edit_instruction`, so no LLM runs. To write the instruction as free-form text, send `prompt="Make the strawberry blue"` instead of `vgl`: that needs an LLM, set before `setup()` with `FIBO_2_TURBO_VGL_LLM` and the credentials of its provider (see [the README](../docs/README.md#the-llm-for-a-free-form-prompt)). Without a size, the edit comes back at about one megapixel, in the photo's aspect ratio.


In [ ]:
from io import BytesIO
from urllib.request import urlopen

from IPython.display import display
from PIL import Image

IMAGE = "https://bria-test-images.s3.us-east-1.amazonaws.com/strawberry.jpg"
VGL = '{"edit_instruction": "Make the strawberry blue"}'

with urlopen(IMAGE, timeout=60) as response:
    photo = Image.open(BytesIO(response.read()))

result = pipeline.execute(Fibo2TurboInput(images=[photo], vgl=VGL, seed=42))
edited_path = outputs_dir / "edited.png"
result.image.save(edited_path)
print(f"Edit {result.image.size}, saved to {edited_path}")
display(result.image)


## 5. Cleanup

Release the GPU memory that the pipeline holds when you are done.


In [ ]:
pipeline.cleanup()
print("Cleanup done.")
